In [5]:
import pandas as pd
import json
import os
from datetime import datetime
import string
import random
import itertools
import uuid
from typing import List, Dict, Optional

def generate_matrix_code() -> str:
    """Generate a matrix code starting with 'SCO' followed by 4 random uppercase letters."""
    return "SCO" + ''.join(random.choice(string.ascii_uppercase) for _ in range(4))

def px_escape(text: str) -> str:
    """Escape quotes in strings for PX format."""
    if isinstance(text, str):
        return text.replace('"', '""')
    return str(text)

def load_metadata(metadata_file: Optional[str]) -> Optional[Dict]:
    """Load metadata from JSON file if provided."""
    if metadata_file and os.path.exists(metadata_file):
        with open(metadata_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def extract_metadata_info(metadata: Dict) -> Dict:
    """Extract relevant metadata fields and infer dimensions."""
    if not metadata or not isinstance(metadata, list) or not metadata:
        return {}

    meta = metadata[0]
    # Extract basic metadata
    info = {
        'title': meta.get('http://purl.org/dc/terms/title', [{}])[0].get('@value', 'Untitled Dataset'),
        'subject_area': meta.get('http://www.w3.org/ns/dcat#theme', [{}])[0].get('@id', 'General').split('/')[-1],
        'description': meta.get('http://purl.org/dc/terms/description', [{}])[0].get('@value', ''),
        'source': meta.get('http://purl.org/dc/terms/publisher', [{}])[0].get('@id', 'Unknown').split('/')[-1],
        'units': meta.get('http://statistics.gov.scot/def/statistical-quality/relevance', [{}])[0].get('@value', 'Unknown')
    }

    # Infer stub and heading columns from metadata structure or related fields
    structure = meta.get('http://purl.org/linked-data/cube#structure', [{}])[0].get('@id', '')
    graph = meta.get('http://publishmydata.com/def/dataset#graph', [{}])[0].get('@id', '')
    
    # Extract potential dimension names from structure or graph URIs
    potential_dims = []
    for uri in [structure, graph]:
        if uri:
            # Extract last part of URI as potential dimension
            dim = uri.split('/')[-1].replace('-', ' ').title()
            potential_dims.append(dim)

    # Additional dimensions from description or themes
    themes = [t.get('@id', '').split('/')[-1].replace('-', ' ').title() 
              for t in meta.get('http://www.w3.org/ns/dcat#theme', [])]
    potential_dims.extend(themes)

    # Clean and deduplicate dimensions
    potential_dims = list(dict.fromkeys([d for d in potential_dims if d]))

    info['potential_dimensions'] = potential_dims
    return info

def load_geojson(input_file: str) -> pd.DataFrame:
    """
    Load GeoJSON file and convert to DataFrame.
    
    This function extracts properties from GeoJSON features and converts them to a DataFrame.
    """
    with open(input_file, 'r', encoding='utf-8') as f:
        geojson_data = json.load(f)
    
    # Extract properties from features
    if geojson_data.get('type') != 'FeatureCollection':
        raise ValueError("Input file must be a GeoJSON FeatureCollection")
    
    # Extract all properties from features
    properties_list = []
    for feature in geojson_data.get('features', []):
        if 'properties' in feature:
            properties_list.append(feature['properties'])
    
    if not properties_list:
        raise ValueError("No feature properties found in the GeoJSON file")
    
    # Convert to DataFrame
    return pd.DataFrame(properties_list)

def define_dimensions(df: pd.DataFrame, dimension_cols: List[str]) -> Dict[str, Dict[str, List[str]]]:
    """Define dimension values and codes dynamically from DataFrame."""
    dimensions = {}
    for col in dimension_cols:
        unique_vals = sorted(df[col].dropna().astype(str).unique())
        dimensions[col] = {
            "values": unique_vals,
            "codes": [f"{i+1:02d}" for i in range(len(unique_vals))]
        }
    return dimensions

def infer_dimensions_from_metadata_and_data(df: pd.DataFrame, metadata_info: Dict) -> tuple[List[str], List[str]]:
    """Infer stub and heading columns from metadata and DataFrame columns."""
    potential_dims = metadata_info.get('potential_dimensions', [])
    df_cols = list(df.columns)

    # Map potential dimensions to actual DataFrame columns
    stub_cols = []
    heading_cols = []

    # Try to match potential dimensions to DataFrame columns
    for dim in potential_dims:
        # Normalize dimension name for matching
        norm_dim = dim.lower().replace(' ', '')
        for col in df_cols:
            norm_col = col.lower().replace(' ', '')
            if norm_dim in norm_col or norm_col in norm_dim:
                if not stub_cols and not heading_cols:
                    stub_cols.append(col)  # First match as stub
                elif col not in stub_cols:
                    heading_cols.append(col)  # Subsequent matches as headings

    # Fallback: if no dimensions matched, use first column as stub, rest as headings (excluding value column)
    if not stub_cols and not heading_cols:
        stub_cols = [df_cols[0]] if df_cols else []
        heading_cols = df_cols[1:]

    return stub_cols, heading_cols

def tidy_to_pxstat(
    input_file: str,
    output_file: Optional[str] = None,
    metadata_file: Optional[str] = None,
    stub_cols: Optional[List[str]] = None,
    heading_cols: Optional[List[str]] = None,
    value_col: Optional[str] = None,
    decimals: Optional[int] = 0,
    agg_method: str = "sum",
    max_dimensions: int = 4,  # Limit number of dimensions to prevent memory errors
    max_values_per_dim: int = 50  # Limit values per dimension
) -> str:
    """
    Convert various data formats (CSV, JSON, GeoJSON) to monolingual PxStat format with dynamic dimensions.

    Parameters:
    -----------
    input_file : str
        Path to input file (CSV, JSON, or GeoJSON).
    output_file : str, optional
        Path to output PX file (default: input_name + ".px").
    metadata_file : str, optional
        Path to metadata JSON file for title, subject, and dimension inference.
    stub_cols : list, optional
        Columns to use as stub dimensions (rows, inferred from metadata if None).
    heading_cols : list, optional
        Columns to use as heading dimensions (columns, inferred from metadata if None).
    value_col : str, optional
        Column name containing the values (will try to auto-detect if None).
    decimals : int, optional
        Number of decimals to use (default: 0 for count data).
    agg_method : str, optional
        Aggregation method for duplicates ("sum" or "mean", default: "sum").
    max_dimensions : int, optional
        Maximum number of dimensions to use (default: 4).
    max_values_per_dim : int, optional
        Maximum number of unique values per dimension (default: 50).
    """
    print(f"Loading data from {input_file}...")
    try:
        # Load input file
        file_ext = os.path.splitext(input_file)[1].lower()
        if file_ext == '.csv':
            df = pd.read_csv(input_file, low_memory=False, dtype_backend="numpy_nullable")
        elif file_ext in ['.json', '.geojson']:
            # Special handling for GeoJSON
            try:
                with open(input_file, 'r', encoding='utf-8') as f:
                    # Read first few characters to check if it's GeoJSON
                    first_chars = f.read(1000)
                    if '"type": "FeatureCollection"' in first_chars or '"type":"FeatureCollection"' in first_chars:
                        print("Detected GeoJSON format...")
                        df = load_geojson(input_file)
                    else:
                        # Try to load as JSON DataFrame
                        df = pd.read_json(input_file, dtype_backend="numpy_nullable")
            except (ValueError, TypeError) as e:
                print(f"Standard JSON loading failed: {str(e)}")
                print("Trying to load as GeoJSON...")
                df = load_geojson(input_file)
        else:
            raise ValueError("Input file must be CSV, JSON, or GeoJSON")

        print(f"Data loaded with {len(df)} rows and {len(df.columns)} columns")
        print(f"Columns: {', '.join(df.columns)}")

        # Load metadata
        metadata = load_metadata(metadata_file)
        meta_info = extract_metadata_info(metadata) if metadata else {}

        # Auto-detect or set value column if not provided
        if not value_col:
            # Try to find numerical column to use as value
            numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
            print(f"Detected numeric columns: {numeric_cols}")
            
            if len(numeric_cols) == 1:
                value_col = numeric_cols[0]
            elif len(numeric_cols) > 1:
                # If multiple numeric columns, prefer a column with 'count', 'value', or 'amount' in its name
                potential_cols = [col for col in numeric_cols if any(x in col.lower() for x in ['count', 'value', 'amount'])]
                if potential_cols:
                    value_col = potential_cols[0]
                else:
                    value_col = numeric_cols[0]
            else:
                # Default: add a count column if no numerical column exists
                df['Count'] = 1
                value_col = 'Count'
            
            print(f"Selected value column: {value_col}")

        # Validate required columns
        if value_col not in df.columns:
            raise ValueError(f"Value column '{value_col}' not found in input data")

        # Infer stub and heading columns if not provided
        if not stub_cols or not heading_cols:
            inferred_stub_cols, inferred_heading_cols = infer_dimensions_from_metadata_and_data(df, meta_info)
            stub_cols = stub_cols or inferred_stub_cols
            heading_cols = heading_cols or inferred_heading_cols

        # Remove value column from potential dimension columns if present
        group_cols = [col for col in (stub_cols + heading_cols) if col != value_col]

        if not group_cols:
            # If no group columns, use categorical or object columns
            cat_cols = list(df.select_dtypes(include=['category', 'object']).columns)
            # Exclude unusable columns (like geometry)
            cat_cols = [col for col in cat_cols if col != 'geometry' and col != value_col]
            group_cols = cat_cols

        if not group_cols:
            raise ValueError("No dimension columns specified or inferred")

        # Limit number of dimensions to prevent memory errors
        if len(group_cols) > max_dimensions:
            print(f"Warning: Limiting dimensions from {len(group_cols)} to {max_dimensions} to prevent memory errors")
            # Choose dimensions with fewer unique values to reduce combinations
            dim_sizes = [(col, df[col].nunique()) for col in group_cols]
            dim_sizes.sort(key=lambda x: x[1])  # Sort by number of unique values
            group_cols = [col for col, _ in dim_sizes[:max_dimensions]]
        
        print(f"Using dimensions: {group_cols}")

        # Validate dimension columns
        missing_cols = [col for col in group_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing dimension columns: {missing_cols}")

        # Create dimensions with stub and heading columns
        stub_cols = [col for col in group_cols if col in stub_cols] or group_cols[:1]
        heading_cols = [col for col in group_cols if col in heading_cols or (col not in stub_cols)]

        # Simplify to needed columns
        df_simple = df[group_cols + [value_col]].copy()

        # Make values numeric
        df_simple[value_col] = pd.to_numeric(df_simple[value_col], errors="coerce")
        if df_simple[value_col].isna().any():
            print(f"Warning: {df_simple[value_col].isna().sum()} non-numeric values in {value_col} set to NaN.")

        # Limit values per dimension to prevent combinatorial explosion
        for col in group_cols:
            if df_simple[col].nunique() > max_values_per_dim:
                print(f"Warning: Column '{col}' has {df_simple[col].nunique()} unique values. Limiting to top {max_values_per_dim}.")
                # Get top values by frequency
                top_values = df_simple[col].value_counts().nlargest(max_values_per_dim).index
                # Replace other values with "Other"
                df_simple.loc[~df_simple[col].isin(top_values), col] = "Other"

        # Convert to string to handle mixed types
        for col in group_cols:
            df_simple[col] = df_simple[col].astype(str)

        # Aggregate duplicates
        print(f"Aggregating data with {agg_method}...")
        df_agg = df_simple.groupby(group_cols, as_index=False).agg({value_col: agg_method})
        print(f"Aggregated to {len(df_agg)} unique combinations")

        # Define dimensions dynamically
        dimensions = define_dimensions(df_simple, group_cols)

        # Calculate expected points and check if it's safe to proceed
        dim_counts = [len(dimensions[col]["values"]) for col in group_cols]
        expected_count = 1
        for count in dim_counts:
            expected_count *= count
            
        print(f"Dimension value counts: {list(zip(group_cols, dim_counts))}")
        print(f"Expected data points: {expected_count}")
        
        if expected_count > 1000000:
            raise ValueError(f"Too many combinations ({expected_count}). Try reducing dimensions or unique values per dimension.")

        # Create a mapping from data - more memory efficient approach
        print("Creating data mapping...")
        data_map = {}
        for _, row in df_agg.iterrows():
            key = tuple(row[col] for col in group_cols)
            data_map[key] = row[value_col]

        # Generate data values using itertools.product more efficiently
        print("Generating data values...")
        data_values = []
        dim_values = [dimensions[col]["values"] for col in group_cols]
        
        # Process combinations in chunks to avoid memory issues
        chunk_size = 100000
        product_iterator = itertools.product(*dim_values)
        
        while True:
            chunk = list(itertools.islice(product_iterator, chunk_size))
            if not chunk:
                break
                
            for combo in chunk:
                value = data_map.get(combo, pd.NA)
                data_values.append(".." if pd.isna(value) else str(int(value) if decimals == 0 else round(value, decimals)))

        # Verify data count
        actual_count = len(data_values)
        if actual_count != expected_count:
            raise ValueError(f"Data count mismatch: expected {expected_count}, got {actual_count}")

        # Metadata
        creation_date = datetime.now().strftime("%Y%m%d %H:%M")
        title = meta_info.get('title', f"Dataset from {os.path.basename(input_file)}")
        subject_area = meta_info.get('subject_area', 'General')
        matrix_code = generate_matrix_code()
        units = px_escape(meta_info.get('units', 'Count'))
        source = meta_info.get('source', 'Unknown')

        header = f"""CHARSET="UTF-16";
AXIS-VERSION="2013";
CREATION-DATE="{creation_date}";
MATRIX="{matrix_code}";
DECIMALS={decimals};
SUBJECT-AREA="{px_escape(subject_area)}";
SUBJECT-CODE="{matrix_code[:4] if len(matrix_code) >= 4 else matrix_code}";
CONTENTS="{px_escape(title)}";
TITLE="{px_escape(title)} - by {', '.join(px_escape(col) for col in group_cols)}";
UNITS="{units}";
STUB="{','.join(f'"{px_escape(col)}"' for col in stub_cols)}";
HEADING="{','.join(f'"{px_escape(col)}"' for col in heading_cols)}";
SOURCE="{px_escape(source)}";
"""

        # VALUES and CODES blocks
        def px_values_and_codes(name: str, dim: Dict) -> str:
            quoted_vals = ",".join(f'"{px_escape(str(v))}"' for v in dim["values"])
            quoted_codes = ",".join(f'"{px_escape(str(c))}"' for c in dim["codes"])
            return f'VALUES("{name}")={quoted_vals};\nCODES("{name}")={quoted_codes};\n'

        meta_parts = "".join(px_values_and_codes(col, dimensions[col]) for col in group_cols)

        # Output filename
        output_file = output_file or os.path.splitext(input_file)[0] + ".px"

        # Write to file
        print(f"Writing {len(data_values)} data points to {output_file}...")
        with open(output_file, "w", encoding="utf-16") as f:
            f.write(header)
            f.write(meta_parts)
            f.write("DATA=\n")
            chunk_size = 1000
            for i in range(0, len(data_values), chunk_size):
                f.write(" ".join(data_values[i:i+chunk_size]) + "\n")
            f.write(";")

        print(f"✅ PX file saved as: {output_file}")
        return output_file

    except Exception as e:
        print(f"❌ Error processing file: {str(e)}")
        raise
if __name__ == "__main__":
    # Example usage for GeoJSON conversion
    CONFIG = {
        "input_file": "Deer Vehicle Collisions.geojson",
        "output_file": "Deer_Vehicle_Collisions.px",
        "value_col": "Count",
        "stub_cols": ["LOCALAUTHO", "DEER_SPECI"],  # Rows
        "heading_cols": ["YEAR", "MONTH"],  # Columns
        "decimals": 0,
        "agg_method": "sum",
        "max_dimensions": 4,  # Allow 4 dimensions
        "max_values_per_dim": 50
    }
    # Run conversion
    print("Running conversion with the following configuration:")
    for key, value in CONFIG.items():
        print(f"{key}: {value}")
    
    # Modify the DataFrame to add a Count column before conversion
    with open(CONFIG["input_file"], 'r', encoding='utf-8') as f:
        geojson_data = json.load(f)
    properties_list = [feature['properties'] for feature in geojson_data.get('features', [])]
    df = pd.DataFrame(properties_list)
    df['Count'] = 1  # Add a count column for each incident
    
    # Save modified DataFrame temporarily to pass to the function
    temp_csv = "temp_deer_collisions.csv"
    df.to_csv(temp_csv, index=False)
    CONFIG["input_file"] = temp_csv  # Update input file to the temporary CSV
    
    try:
        tidy_to_pxstat(**CONFIG)
    finally:
        # Clean up temporary file
        if os.path.exists(temp_csv):
            os.remove(temp_csv)

Running conversion with the following configuration:
input_file: Deer Vehicle Collisions.geojson
output_file: Deer_Vehicle_Collisions.px
value_col: Count
stub_cols: ['LOCALAUTHO', 'DEER_SPECI']
heading_cols: ['YEAR', 'MONTH']
decimals: 0
agg_method: sum
max_dimensions: 4
max_values_per_dim: 50
Loading data from temp_deer_collisions.csv...
Data loaded with 16383 rows and 16 columns
Columns: REF, INC_DATE, OS12FIGREF, OS_EASTING, OS_NORTHIN, DEER_SPECI, LOCALAUTHO, ROAD_NO, YEAR, MONTH, PERIOD, CORE_OROTH, ONTRUNK_OR, SPECPROJ_P, GRIDACCURA, Count
Using dimensions: ['LOCALAUTHO', 'DEER_SPECI', 'YEAR', 'MONTH']
Aggregating data with sum...
Aggregated to 3645 unique combinations
Dimension value counts: [('LOCALAUTHO', 32), ('DEER_SPECI', 5), ('YEAR', 11), ('MONTH', 12)]
Expected data points: 21120
Creating data mapping...
Generating data values...
Writing 21120 data points to Deer_Vehicle_Collisions.px...
✅ PX file saved as: Deer_Vehicle_Collisions.px
